### ESQUEMA

1. Cargar y explorar datos: pandas

2. Preprocesado de texto: re, NLTK

3. Vectorización: scikit-learn (TF-IDF / CountVectorizer)

4. Modelo: scikit-learn (Naive Bayes, y luego otros)

5. Evaluación: scikit-learn (metrics, confusion matrix)

6. Mejoras. Desbalance de clases: imbalanced-learn

7. Bonus. Modelo más potente: transformers (BERT fine-tuning)

In [12]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("../data/english_comments.csv")

df.head(3)

,Comment,Sentiment
0,lets not forget that apple pay in 2014 require...,neutral
1,here in nz 50 of retailers dont even have cont...,negative
2,i will forever acknowledge this channel with t...,positive


In [ ]:
# =======================================================================================================
# VECTORIZACIÓN
# =======================================================================================================

# En este caso se usará TF-IDF (Term Frequency — Inverse Document Frequency):
# TF: cuántas veces aparece la palabra en ese comentario
# IDF: logaritmo inverso de en cuántos documentos aparece
# Penaliza las palabras que aparecen en muchos documentos. A las palabras raras pero significativas como "horrible" o "amazing" les da más peso.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

def vectorization(X_data, y_data):
    # stratify=y es importante: mantiene la proporción de clases en train y test
    X_train, X_test, y_train, y_test = train_test_split(
        X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
    )

    # Creando vectorizador
    # ignora palabras que aparecen < 3 veces
    # límite de vocabulario
    # unigramas y bigramas
    vectorizer = TfidfVectorizer(min_df=3, max_features=100000, ngram_range=(1,2))

    # Vectorizando los comentarios
    # fit_transform para train porque el modelo aprende de estos comentarios: qué palabras existen, qué posición ocupa cada una en el vector, cuál es el IDF de cada una.
    # transform para test porque sólo necesitamos vectorizarlos (en base a lo aprendido para train). Si usamos fit volveríamos a reconstruir el modelo en base a los datos de test
    X_train_v = vectorizer.fit_transform(X_train)
    X_test_v = vectorizer.transform(X_test)

    # usar fit con datos de entrenamiento ensucia el aprendizaje (data leakage)

    return X_train_v, X_test_v, y_train, y_test


In [ ]:
# =======================================================================================================
# MODELO NAIVE BAYES
# =======================================================================================================

# Es el modelo natural para vectores de conteos y TF-IDF porque está diseñado para features que representan frecuencias

from sklearn.naive_bayes import MultinomialNB

clf = MultinomialNB(alpha = 1)

def train_and_evaluate(model, X_train, X_test, y_train, y_test):
    

    model.fit(X_train, y_train)

    results = model.predict(X_test)

    accuracy = model.score(X_test, y_test)

    return results, accuracy



In [ ]:
# =======================================================================================================
# EVALUACIÓN
# =======================================================================================================

# classification_report — da tres métricas por cada clase:
# Precision: de los que predije como positivos, ¿cuántos lo eran realmente?
# Recall: de los que eran realmente positivos, ¿cuántos detecté?
# F1-score: media armónica de las dos anteriores. Es la métrica en la que fijarse cuando hay desbalance.

import matplotlib as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def evaluation(y_test, y_pred, target_names = ["Positivo", "Neutro", "Negativo"]):
    report_dict = classification_report(y_test, y_pred, target_names=target_names, output_dict=True)

    conf_matrix = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(conf_matrix, display_labels=target_names)

    print(classification_report(y_test, y_pred, target_names=target_names))
    disp.plot()
    plt.show()

    return report_dict
